# СИМА — Интерактивное построение ЦМР/DTM/DSM

Ноутбук для интерактивного расчёта с настраиваемыми параметрами.
Все параметры инкапсулированы в dataclass-конфигах.
Сглаживание с интерполяцией дырок, склоны по сглаженной DTM, без экстраполяции краёв.

**Параметры можно менять в ячейке 2 и перезапускать расчёт.**

In [ ]:
import sys, os
from pathlib import Path
import rasterio
import numpy as np
import laspy
import matplotlib.pyplot as plt

backend = Path('/Users/sergeyzay/Documents/НЕДРА/СИМА/sima-web/backend')
for pkg in ['packages/sima-dem-core/src', 'packages/sima-dem-ground/src',
            'packages/sima-dem-dsm/src', 'packages/sima-dem-pipeline/src']:
    sys.path.insert(0, str(backend / pkg))

from sima_dem_ground.ground import GroundProcessing, SMRFConfig, FillConfig
from sima_dem_dsm.dsm import DSMBuilder, DSMConfig
from sima_dem_core.curvature import CurvatureProcessing
from sima_dem_core.raster.smooth import gauss_smooth
from sima_dem_core.raster.tpi import calculate_tpi, TPIConfig
from sima_dem_core.check_classification import CheckClassification

print('Импорт готов')

In [ ]:
# === ПАРАМЕТРЫ ===

DATASET = 'demo'  # 'demo' или 'test'

if DATASET == 'demo':
    LAS_PATH = '/Users/sergeyzay/Documents/НЕДРА/СИМА/23_04_12_digital_elevation_1-46-315/demo_data/pt000100.las'
    TIF_PATH = '/Users/sergeyzay/Documents/НЕДРА/СИМА/23_04_12_digital_elevation_1-46-315/demo_data/00000100.tif'
    REFERENCE_DSM = None
elif DATASET == 'test':
    LAS_PATH = '/Users/sergeyzay/Documents/НЕДРА/СИМА/test_data/P-42-041-239-g_ground_TLO.las'
    TIF_PATH = '/Users/sergeyzay/Documents/НЕДРА/СИМА/test_data/P-42-041-239-g.tif'
    REFERENCE_DSM = '/Users/sergeyzay/Documents/НЕДРА/СИМА/test_data/P-42-041-239-g_DSM.tif'

OUTPUT_DIR = str(backend / 'output' / f'notebook_{DATASET}')
os.makedirs(OUTPUT_DIR, exist_ok=True)

RESOLUTION = 1.0
GAUSS_SIGMA = 2.0
GAUSS_WINDOW = 5
BUILD_DSM = True
BUILD_DTM = True
BUILD_SLOPE = True
BUILD_ASPECT = True
BUILD_TPI = True

print(f'Датасет: {DATASET}, LAS: {LAS_PATH}, Выход: {OUTPUT_DIR}')

In [ ]:
crs_source = REFERENCE_DSM if REFERENCE_DSM else TIF_PATH
with rasterio.open(crs_source) as src:
    CRS = src.crs.to_wkt()
print(f'CRS: {CRS[:80]}...')

In [ ]:
if REFERENCE_DSM:
    sys.path.insert(0, str(backend))
    from tests.fixtures.restore_las import restore_absolute_las
    restored = str(Path(OUTPUT_DIR) / 'restored_absolute.las')
    restore_absolute_las(LAS_PATH, REFERENCE_DSM, restored)
    LAS_PATH = restored
    print(f'Восстановлен: {restored}')
else:
    print('Восстановление не требуется')

In [ ]:
las = laspy.read(LAS_PATH)
cls = np.asarray(las.classification)
print(f'Точек: {len(las.points):,}')
print(f'Классы: {sorted(set(cls))}')
print(f'Z: {np.min(las.z):.1f} – {np.max(las.z):.1f}')
print(f'X: {np.min(las.x):.1f} – {np.max(las.x):.1f}')
print(f'Y: {np.min(las.y):.1f} – {np.max(las.y):.1f}')

In [ ]:
if BUILD_DTM:
    gp = GroundProcessing(
        output=OUTPUT_DIR,
        resolution=RESOLUTION,
        crs=CRS,
        interpolate=True,
        save_ground_las=False,
        smrf=SMRFConfig(slope=0.2, window=16, threshold=0.45, scalar=1.2),
        fill=FillConfig(fill_holes=True, max_search_distance=100),
    )
    gp.get_raster(LAS_PATH, crs_wkt=CRS)
    dtm_path = gp.raster[0]
    print(f'DTM: {dtm_path}')
else:
    dtm_path = None
    print('DTM пропущен')

In [ ]:
if BUILD_DSM:
    builder = DSMBuilder(
        output=OUTPUT_DIR,
        crs=CRS,
        config=DSMConfig(
            resolution=RESOLUTION,
            output_type='max',
            interpolate=True,
            fill_holes=True,
            max_search_distance=100,
        ),
    )
    dsm_path = builder.build(LAS_PATH, crs_wkt=CRS)
    print(f'DSM: {dsm_path}')
else:
    dsm_path = None
    print('DSM пропущен')

In [ ]:
if BUILD_DTM and GAUSS_SIGMA > 0:
    stem = Path(dtm_path).stem.replace('_dem', '')
    smoothed_path = str(Path(OUTPUT_DIR) / (stem + '_dem_smooth.tif'))
    gauss_smooth(
        dtm_path, smoothed_path,
        sigma=GAUSS_SIGMA * RESOLUTION,
        order=0, window_size=GAUSS_WINDOW,
        fill_holes=True, max_search_distance=100,
    )
    print(f'Сглаженная: {smoothed_path}')
else:
    smoothed_path = None

In [ ]:
base_raster = smoothed_path if smoothed_path else dtm_path

cp = CurvatureProcessing()
slope_path = aspect_path = None
if BUILD_SLOPE and base_raster:
    slope_path = cp.calculate_slope(base_raster, CRS, RESOLUTION, RESOLUTION, OUTPUT_DIR)
    print(f'Уклоны: {slope_path}')
if BUILD_ASPECT and base_raster:
    aspect_path = cp.calculate_aspect(base_raster, CRS, RESOLUTION, RESOLUTION, OUTPUT_DIR)
    print(f'Экспозиции: {aspect_path}')

In [ ]:
if BUILD_TPI and base_raster:
    tpi_path = calculate_tpi(
        base_raster, CRS, OUTPUT_DIR, RESOLUTION, 10.0,
        config=TPIConfig(radii_m=[270, 810, 2430], res=10.0),
    )
    print(f'TPI: {tpi_path}')
else:
    tpi_path = None

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

plots = []
if dtm_path: plots.append(('DTM (ЦМР)', dtm_path, 'terrain'))
if dsm_path: plots.append(('DSM (ЦМД)', dsm_path, 'terrain'))
if slope_path: plots.append(('Уклоны (°)', slope_path, 'hot'))
if aspect_path: plots.append(('Экспозиции (°)', aspect_path, 'hsv'))

for i, (title, path, cmap) in enumerate(plots[:4]):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(float)
        if src.nodata is not None:
            arr = np.where(arr == src.nodata, np.nan, arr)
    im = axes[i].imshow(arr, cmap=cmap)
    axes[i].set_title(title)
    plt.colorbar(im, ax=axes[i], shrink=0.6)

for j in range(len(plots), 4):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig(str(Path(OUTPUT_DIR) / 'overview.png'), dpi=150)
plt.show()

In [ ]:
if REFERENCE_DSM and dtm_path:
    with rasterio.open(dtm_path) as src:
        built = src.read(1).astype(float)
        b_nd = src.nodata
    with rasterio.open(REFERENCE_DSM) as src:
        ref = src.read(1).astype(float)
        r_nd = src.nodata
    min_h = min(built.shape[0], ref.shape[0])
    min_w = min(built.shape[1], ref.shape[1])
    built, ref = built[:min_h,:min_w], ref[:min_h,:min_w]
    valid = (built != b_nd) & (ref != r_nd)
    diff = np.abs(built[valid] - ref[valid])
    rmse = np.sqrt(np.mean(diff**2))
    mean_ref = np.mean(ref[valid])
    rel_err = rmse / mean_ref
    print(f'RMSE: {rmse:.4f} м')
    print(f'Mean эталона: {mean_ref:.4f} м')
    print(f'Относительная ошибка: {rel_err:.4%}')
    print(f'Требование < 5%: {"✓ ПРОЙДЕН" if rel_err < 0.05 else "✗ НЕ ПРОЙДЕН"}')

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, title, path in [(axes[0], 'Построено', dtm_path), (axes[1], 'Эталон', REFERENCE_DSM)]:
        with rasterio.open(path) as src:
            arr = src.read(1).astype(float)
            if src.nodata is not None: arr = np.where(arr == src.nodata, np.nan, arr)
        ax.imshow(arr, cmap='terrain')
        ax.set_title(title)
    diff_map = np.where(valid, np.abs(built - ref), np.nan)
    im = axes[2].imshow(diff_map, cmap='Reds')
    axes[2].set_title(f'Разница (RMSE={rmse:.2f} м)')
    plt.colorbar(im, ax=axes[2])
    plt.tight_layout()
    plt.show()
else:
    print('Сравнение с эталоном недоступно')

In [ ]:
print(f'Выходные файлы ({OUTPUT_DIR}):')
for f in sorted(Path(OUTPUT_DIR).glob('*')):
    if f.suffix in ('.tif', '.las', '.png'):
        size = f.stat().st_size / 1024 / 1024
        print(f'  {f.name}  ({size:.1f} MB)')